# Capstone - Ranking Signal Analysis
### Which signals actually predict content decline?
**Haneef Aderolu · 2026 · Lane 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HaneefAderolu/ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook mirrors the deployed research paper. It runs the full pipeline end-to-end and
generates every figure and metric the paper embeds. Run all cells top-to-bottom.

In [ ]:
# Cell 0 - Setup
import os, json
if not os.path.exists('ml-internship-starter'):
    os.system('git clone https://github.com/HaneefAderolu/ml-internship-starter.git')
os.chdir('ml-internship-starter')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)
os.makedirs('docs/img', exist_ok=True)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Proxy label - same definition used in W04, W05, W07
df['label'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] >= 500) &
    (df['content_age_days'] >= 180)
).astype(int)

df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)
df['has_position']       = (df['avg_position'] > 0).astype(int)
df['has_word_count']     = df['word_count'].notna().astype(int)
df['word_count_filled']  = df['word_count'].fillna(0)

features = [
    'impressions_90d','days_with_impressions','days_with_sessions',
    'avg_position_clean','has_position','ctr','engagement_rate',
    'scroll_rate','word_count_filled','has_word_count',
    'content_age_days','days_since_last_update','search_volume',
    'competition','sessions_90d','pageviews_90d',
]

X      = df[features].copy().fillna(df[features].median())
y      = df['label'].copy()
groups = df['client_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

print(f"Dataset: {len(df):,} pages | {df['client_id'].nunique()} clients")
print(f"Label base rate: {y.mean():.1%}")
print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
print("Model ready.")

## 1. Question

**Research question:**
Which measurable page signals most reliably separate content that is declining in search
visibility from content that is stable - across a diverse portfolio of 30,000 pages?

**Decision this supports:**
A content editor has limited hours each week. Of 30,000 pages, they need to know which
20–100 to prioritise for review. This analysis builds a ranked queue that answers exactly that.

**Why this is a data problem:**
A simple age rule ("flag all pages older than 180 days") would flag 17,986 pages.
Of those, 16.4% are still trending upward - the rule would waste editorial time
on healthy content. A model that combines age, visibility, engagement, and position
can separate the genuinely at-risk pages from the merely old ones.

In [ ]:
# Section 1 - Show why a simple rule fails
stale = df[df['content_age_days'] >= 180]
stale_up = stale[stale['trend_direction'] == 'up']

print("Why the age rule alone fails:")
print(f"  Pages stale (180+ days):             {len(stale):,}")
print(f"  Of those, STILL trending up:         {len(stale_up):,} ({len(stale_up)/len(stale):.1%}) - false alarms")
print(f"  Pages declining but under 90 days:   {((df['trend_direction']=='down') & (df['content_age_days']<90)).sum():,} - missed by age rule")
print(f"\nBase rate (random queue would achieve): {y.mean():.1%}")

## 2. Data

**Source:** FlyRank ML Internship dataset - `data/raw/content_refresh_anonymized.csv`

**Size:** 30,000 pages · 32 client accounts · 44 columns

**Time window:** Trailing 90-day snapshot. Each row summarises a content page's
search performance over the 90 days ending at the export date.

**What was excluded and why:**
- `trend_direction` and `trend_pct` - these ARE the label source. Using them as features
  would be circular leakage.
- `content_id` and `client_id` - identifiers, not signals. Used only for grouping.
- Pages with `avg_position = 0` - zero means "no GSC data recorded", not rank zero.
  Treated as missing and excluded from position-based features.
- June 2026 warehouse partition - sealed test month, never used for label logic.

**Public safety:** all client identifiers are anonymised hashes (e.g. `client_6208ef0f77`).
No raw URLs, private queries, or client names appear anywhere.

In [ ]:
# Section 2 - Data summary
print("Dataset summary:")
print(f"  Total pages:          {len(df):,}")
print(f"  Client accounts:      {df['client_id'].nunique()}")
print(f"  Columns available:    {len(df.columns)}")
print(f"  Features used:        {len(features)}")
print(f"  Positive labels:      {y.sum():,} ({y.mean():.1%})")
print(f"\nExcluded as label-derived:")
print(f"  trend_direction:      {df['trend_direction'].value_counts().to_dict()}")
print(f"\nExcluded as identifiers:")
print(f"  Unique content_ids:   {df['content_id'].nunique():,}")
print(f"  Unique client_ids:    {df['client_id'].nunique()}")

## 3. Methodology

**Proxy label definition:**
A page is labelled 1 (worth prioritising for review) if ALL THREE conditions hold:
- `trend_direction == 'down'` - traffic is declining
- `impressions_90d >= 500` - page is visible enough to matter
- `content_age_days >= 180` - content is at least 6 months old

This is a proxy, not a ground truth. "Declining + visible + old" is our best
available signal for "needs editorial attention" given the available data.
A page labelled 1 should be reviewed by a human - not automatically rewritten.

**Features (16 total):**
Search visibility: `impressions_90d`, `days_with_impressions`, `days_with_sessions`, `ctr`
Positional: `avg_position_clean`, `has_position`
Engagement: `engagement_rate`, `scroll_rate`
Content attributes: `word_count_filled`, `has_word_count`, `content_age_days`, `days_since_last_update`
Demand context: `search_volume`, `competition`
Traffic: `sessions_90d`, `pageviews_90d`

**Baseline (W04 rule):**
Score = position_score + maturity_score + engagement_score
Hand-written thresholds. No fitting. Pages ranked by this score.

**Model:** Random Forest, 200 trees, max_depth=6, class_weight='balanced', random_state=42.
`class_weight='balanced'` because the positive class is only 17.9% - without it the model
would predict "no review needed" for everything and be right 82% of the time.

**Validation design:** GroupShuffleSplit by `client_id`, 75/25 train/test.
All pages from a given client go entirely into train OR entirely into test.
This prevents client-level leakage - the model must generalise to unseen clients.

**Leakage checks:**
✓ No `trend_direction` or `trend_pct` in features (label source)
✓ No `gsc_clicks` in features (clicks/impressions = CTR, circular with impressions)
✓ No future-window data used

In [ ]:
# Section 3 - Validate the split design
train_clients = df['client_id'].iloc[train_idx].unique()
test_clients  = df['client_id'].iloc[test_idx].unique()

overlap = set(train_clients) & set(test_clients)
print("Split validation:")
print(f"  Train clients: {len(train_clients)} | Test clients: {len(test_clients)}")
print(f"  Client overlap (must be 0): {len(overlap)}")
print(f"  Train positive rate: {y_train.mean():.3f}")
print(f"  Test positive rate:  {y_test.mean():.3f}")
print(f"  Base rate (full):    {y.mean():.3f}")
print("\nLeakage check - features used:")
for f in features:
    print(f"  ✓ {f}")
print("\nLeakage check - excluded:")
for col in ['trend_direction','trend_pct','gsc_clicks','content_id','client_id']:
    print(f"  ✗ {col}")

## 4. Results (vs baseline)

**Metric:** Precision@K - of the top K pages the model ranks highest,
what fraction actually have label = 1?

This is the right metric because editors look at a ranked queue.
They need the top 20–100 pages to be real opportunities, not noise.

| Method | P@20 | P@50 | P@100 | Base rate |
|---|---|---|---|---|
| Random (no model) | 0.179 | 0.179 | 0.179 | 0.179 |
| Baseline (W04 rule) | 0.250 | 0.220 | 0.210 | 0.179 |
| Random Forest | **0.650** | **0.540** | **0.530** | 0.179 |

The Random Forest achieved 2.6× the precision of a random queue at K=20,
and 2.6× the precision of the hand-written baseline rule.

**What the model learned:**
- `content_age_days` (Gini importance: 0.41) - the strongest split driver
- `impressions_90d` (Gini importance: 0.24) - volume of past visibility
- `days_with_impressions` (Gini importance: 0.14) - page maturity proxy

Permutation importance confirms `impressions_90d` is most predictive
(shuffling it causes the largest score drop).

**Error analysis:**
False positives are pages that are old and visible but not actually declining -
the label requires ALL THREE conditions; the model can mistake "old + visible" for "at risk".
False negatives are pages sitting right at the boundary of all three label thresholds.

In [ ]:
# Section 4 - Results table + figures

def precision_at_k(sorted_df, k, label_col='label'):
    return sorted_df.head(k)[label_col].mean()

# Baseline
test_df = X_test.copy()
test_df['label'] = y_test.values

def baseline_score(row):
    score = 0
    pos = row['avg_position_clean']
    if pd.notna(pos):
        if pos <= 10:   score += 3
        elif pos <= 20: score += 2
        elif pos <= 50: score += 1
    if row['days_with_impressions'] >= 25: score += 2
    elif row['days_with_impressions'] >= 15: score += 1
    eng = row['engagement_rate']
    if eng < 20 and row['impressions_90d'] > 0: score += 2
    elif eng < 50: score += 1
    return score

test_df['baseline_score'] = test_df.apply(baseline_score, axis=1)
base_sorted = test_df.sort_values('baseline_score', ascending=False)
test_df['rf_score'] = rf_probs
rf_sorted = test_df.sort_values('rf_score', ascending=False)

baseline_p20  = precision_at_k(base_sorted, 20)
baseline_p50  = precision_at_k(base_sorted, 50)
baseline_p100 = precision_at_k(base_sorted, 100)
rf_p20  = precision_at_k(rf_sorted, 20)
rf_p50  = precision_at_k(rf_sorted, 50)
rf_p100 = precision_at_k(rf_sorted, 100)

print(f"{'Method':<22} {'P@20':>8} {'P@50':>8} {'P@100':>8}")
print(f"{'─'*50}")
print(f"{'Base rate':<22} {y_test.mean():>8.3f} {y_test.mean():>8.3f} {y_test.mean():>8.3f}")
print(f"{'Baseline (W04)':<22} {baseline_p20:>8.3f} {baseline_p50:>8.3f} {baseline_p100:>8.3f}")
print(f"{'Random Forest':<22} {rf_p20:>8.3f} {rf_p50:>8.3f} {rf_p100:>8.3f}")

# Figure 1 - Results bar chart
ks = ['P@20', 'P@50', 'P@100']
base_vals = [baseline_p20, baseline_p50, baseline_p100]
rf_vals   = [rf_p20, rf_p50, rf_p100]
base_rate = [y_test.mean()] * 3

x = range(len(ks))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - 0.25 for i in x], base_rate, 0.22, label='Base rate (random)', color='#8892a4', alpha=0.7)
ax.bar([i - 0.02 for i in x], base_vals, 0.22, label='Baseline (W04 rule)', color='#fb923c', alpha=0.85)
ax.bar([i + 0.23 for i in x], rf_vals,   0.22, label='Random Forest',       color='#4f8ef7', alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(ks, fontsize=12)
ax.set_ylim(0, 1); ax.set_ylabel('Precision@K'); ax.set_title('Model vs Baseline - Precision@K')
ax.legend(); ax.axhline(y_test.mean(), color='#8892a4', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig('work/figures/fig1_results.png', dpi=150, bbox_inches='tight')
plt.savefig('docs/img/fig1_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ fig1_results.png saved")

# Figure 2 - Feature importance
fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 5))
fi[::-1].plot(kind='barh', ax=ax, color='#4f8ef7', edgecolor='none')
ax.set_xlabel('Gini importance'); ax.set_title('Top 10 feature importances - Random Forest')
plt.tight_layout()
plt.savefig('work/figures/fig2_importance.png', dpi=150, bbox_inches='tight')
plt.savefig('docs/img/fig2_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ fig2_importance.png saved")

## 5. Limitations

**1. Proxy label, not ground truth.**
Our label is derived from a rule: declining + visible + old. We do not know whether refreshing
the flagged pages would actually recover their traffic. This analysis is observational -
it identifies patterns associated with decline, not causes.

**2. Cross-sectional snapshot.**
The starter CSV is a 90-day snapshot, not a time series. We cannot track whether a page
recovered after a refresh. Causal claims ("refreshing will increase traffic") cannot be
made from this data.

**3. Seasonal content is mislabelled.**
A Christmas gift guide that declines in January is correctly declining but does not need
a refresh - it needs to be left alone. The model cannot distinguish seasonal from
permanent decline without topic-level data we do not have.

**4. Uneven client history.**
Some clients have 14 months of indexed history; others joined recently. The `content_age_days`
signal behaves differently per client. Global thresholds applied to this mixed panel
may overweight long-standing clients.

**5. Test set is 8 clients.**
The grouped split produces a test set of 8 clients. Precision@K estimates carry uncertainty
at this scale. The numbers are directional, not a guarantee of future performance
on new clients.

**6. No intervention data.**
The dataset contains no record of past refreshes and their outcomes. We cannot measure
whether our recommendations would work - only whether the model identifies the pages
our proxy label defines as at-risk.

In [ ]:
# Section 5 - Quantify limitations
print("Limitation evidence:")
print(f"  Test clients: {df['client_id'].iloc[test_idx].nunique()} (small - estimates are directional)")
print(f"  Seasonal risk: {(df['trend_pct'] > 500).sum():,} pages with trend_pct > 500% (spike/crash pattern)")
print(f"  New content (<90 days): {(df['content_age_days']<90).sum():,} pages - unreliable age signal")
print(f"  Proxy label base rate: {y.mean():.1%} - label is a rule, not observed ground truth")
print(f"  False positives at P@20: ~7 per batch of 20 (1 - 0.65 = 0.35)")

## 6. Ranked recommendations

The action playbook turns model scores into a human-readable priority queue.
Six reason codes describe why each page was flagged. Four action labels tell
an editor what to do.

| Reason code | Condition | Action |
|---|---|---|
| OLD_HIGH_VOLUME | 365+ days old, 1,000+ impressions | PRIORITISE_REVIEW |
| STALE_WEAK_POSITION | 180+ days old, visible, position > 20 | PRIORITISE_REVIEW |
| STALE_LOW_ENGAGEMENT | 180+ days old, visible, engagement < 20% | SCHEDULE_REVIEW |
| HIGH_VOLUME_ACTIVE | 2,000+ impressions, 70+ active days | MONITOR |
| STALE_VISIBLE | 180+ days old, visible, no stronger signal | SCHEDULE_REVIEW |
| MODERATE_RISK | No strong signal | DEPRIORITISE |

**Observed in data:**
- 9,799 pages carry PRIORITISE_REVIEW (32.7% of all scored pages)
- Top 20 pages have mean content age 275 days and mean 4,653 impressions
- P@20 = 0.65: 13 of every 20 top-ranked pages match our proxy label

**What should never be automated:**
Publishing content based on scores · Deleting DEPRIORITISE pages · Using scores as writer KPIs

In [ ]:
# Section 6 - Build and display the ranked queue
X_full       = df[features].copy().fillna(df[features].median())
rf_probs_full = rf.predict_proba(X_full)[:, 1]

queue = df[['content_id','client_id','impressions_90d','avg_position_clean',
            'content_age_days','engagement_rate','days_with_impressions',
            'ctr','trend_direction','label']].copy()
queue['rf_score'] = rf_probs_full
queue = queue.sort_values('rf_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

def reason_code(row):
    age, imp = row['content_age_days'], row['impressions_90d']
    pos, eng  = row['avg_position_clean'], row['engagement_rate']
    days      = row['days_with_impressions']
    if age >= 365 and imp >= 1000:                                     return 'OLD_HIGH_VOLUME'
    elif age >= 180 and imp >= 500 and (pd.isna(pos) or pos > 20):    return 'STALE_WEAK_POSITION'
    elif age >= 180 and imp >= 500 and eng < 20:                      return 'STALE_LOW_ENGAGEMENT'
    elif imp >= 2000 and days >= 70:                                   return 'HIGH_VOLUME_ACTIVE'
    elif age >= 180 and imp >= 500:                                    return 'STALE_VISIBLE'
    else:                                                              return 'MODERATE_RISK'

def action_label(score):
    if score >= 0.6:   return 'PRIORITISE_REVIEW'
    elif score >= 0.4: return 'SCHEDULE_REVIEW'
    elif score >= 0.2: return 'MONITOR'
    else:              return 'DEPRIORITISE'

queue['reason_code'] = queue.apply(reason_code, axis=1)
queue['action']      = queue['rf_score'].apply(action_label)

print("Action distribution:")
print(queue['action'].value_counts().to_string())
print("\nTop 10 pages:")
print(queue[['rank','impressions_90d','avg_position_clean','content_age_days',
             'rf_score','reason_code','action']].head(10).to_string())

# Figure 3 - Reason codes
fig, ax = plt.subplots(figsize=(9, 4))
queue['reason_code'].value_counts().plot(kind='bar', ax=ax, color='#fb923c', edgecolor='none')
ax.set_xlabel('Reason code'); ax.set_ylabel('Number of pages')
ax.set_title('Pages by reason code'); plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('work/figures/fig3_reason_codes.png', dpi=150, bbox_inches='tight')
plt.savefig('docs/img/fig3_reason_codes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ fig3_reason_codes.png saved")

## 7. Artifacts the paper embeds

All outputs are written by this notebook so the paper can embed them.
The CSV stays out of git (CI leak-guard blocks data files).
Figures and metrics JSON are committed.

In [ ]:
# Section 7 - Write all artifacts
queue.to_csv('work/outputs/w07_action_queue.csv', index=False)
print(f"✓ work/outputs/w07_action_queue.csv  ({len(queue):,} rows, NOT committed)")

metrics = {
    "total_pages_ranked":       int(len(queue)),
    "label_base_rate":          round(float(y.mean()), 3),
    "precision_at_20_baseline": round(float(baseline_p20),  3),
    "precision_at_50_baseline": round(float(baseline_p50),  3),
    "precision_at_100_baseline":round(float(baseline_p100), 3),
    "precision_at_20_rf":       round(float(rf_p20),  3),
    "precision_at_50_rf":       round(float(rf_p50),  3),
    "precision_at_100_rf":      round(float(rf_p100), 3),
    "action_distribution":      queue['action'].value_counts().to_dict(),
    "reason_code_distribution": queue['reason_code'].value_counts().to_dict(),
    "top20_mean_rf_score":      round(float(queue.head(20)['rf_score'].mean()), 3),
    "top20_mean_age_days":      round(float(queue.head(20)['content_age_days'].mean()), 0),
    "top20_mean_impressions":   round(float(queue.head(20)['impressions_90d'].mean()), 0),
    "train_clients":            int(df['client_id'].iloc[train_idx].nunique()),
    "test_clients":             int(df['client_id'].iloc[test_idx].nunique()),
}
with open('work/outputs/capstone_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ work/outputs/capstone_metrics.json  (committed)")
print()
print(json.dumps(metrics, indent=2))

## Self-check

- [ ] All 7 sections filled - markdown reasoning AND code that backs it
- [ ] Notebook runs top-to-bottom with no errors (Runtime → Run all)
- [ ] No client names, private URLs, or raw queries anywhere
- [ ] Careful claim language throughout: observed, associated, directional, decision-support
- [ ] All three figures saved to both `work/figures/` and `docs/img/`
- [ ] Metrics JSON saved to `work/outputs/capstone_metrics.json`
- [ ] `submission/paper_url.txt` contains the live deployed URL
- [ ] Paper at deployed URL contains all 9 sections including Abstract and Acknowledgments
- [ ] FlyRank data credit with https://flyrank.ai link present at bottom of paper

---

### 5-minute demo outline (ML-12)
1. Show the deployed paper URL - "here is the research question"
2. Show the results table - "here is what the model achieved vs the baseline"
3. Show the reason codes - "here is what a content editor actually does with it"
4. Show one notebook - "here is how it was built and validated"
5. Name one limitation - "here is what I would do next with more time"

### Social post
> Built a ranking signal analysis on 30,000 real search pages across 32 clients.
> A Random Forest scored Precision@20 = 0.65 - 2.6× better than the hand-written baseline.
> Content age + impression volume are the dominant signals.
> Paper + code: [your GitHub Pages URL]
> #MachineLearning #SEO #ContentStrategy

### Employer 3-sentencer
I built a content priority scoring system on 30,000 real Google Search Console pages
from 32 clients, using a Random Forest trained on 16 signals and validated on unseen clients.
The model ranked at-risk pages at Precision@20 = 0.65, outperforming a hand-written rule
baseline by 2.6×. The output is a ranked review queue used as decision-support for editorial
teams, not an automated system - reflecting an honest understanding of what the data can and
cannot claim.